# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 请填写  
**成员：** 请填写  
**专题方向：** A / B / C / D / E

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "5组"
MEMBERS = ["周睿"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

小组： 5组 ['周睿']
专题： A
输入： c:\Users\Lenovo\Desktop\新建文件夹 (2)\output\day04_project\ecommerce_customer_cleaned.csv
输出： c:\Users\Lenovo\Desktop\新建文件夹 (2)\output\day05_student\5组


### 检查点0

- [ ] 已填写组号、成员和专题；
- [ ] Notebook文件名包含组号；
- [ ] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [2]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# TODO 2：输出shape、前5行和字段类型
print("数据形状:", df.shape)
print("\n前5行:")
print(df.head())
print("\n字段类型:")
print(df.dtypes)

# TODO 3：计算以下验收结果
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[["CustomerID", "Churn", "OrderCount", "TenureGroup", "PreferedOrderCat"]].isnull().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}
validation

数据形状: (5630, 22)

前5行:
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1    4.00         Mobile Phone         3             6.00   
1       50002      1    9.00         Mobile Phone         1             8.00   
2       50003      1    9.00         Mobile Phone         1            30.00   
3       50004      1    0.00         Mobile Phone         3            15.00   
4       50005      1    0.00         Mobile Phone         1            12.00   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female            3.00                         3   
1                  UPI    Male            3.00                         4   
2           Debit Card    Male            2.00                         4   
3           Debit Card    Male            2.00                         4   
4          Credit Card    Male            3.00                         3   

     PreferedOrderCat  SatisfactionScor

{'行数': 5630, '列数': 22, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [3]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话填写：  
____________________________________________________________

## 任务2：公共基础指标（必做）

In [4]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数
overall_metrics = pd.DataFrame({
    "指标": [
        "总用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现金额",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数",
    ],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ],
})

display(overall_metrics)


,指标,数值
0,总用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现金额,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [5]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [6]:
# TODO：填写你的分组字段
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "nunique"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
).reset_index()

# TODO：重置索引、排序并展示
tenure_order = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
segment_analysis["_sort"] = segment_analysis[segment_field].apply(lambda x: tenure_order.index(x))
segment_analysis = segment_analysis.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
segment_analysis["占比"] = (segment_analysis["用户数"] / segment_analysis["用户数"].sum()).round(4)
segment_analysis = segment_analysis[[segment_field, "用户数", "占比", "流失率", "平均订单数", "平均返现", "平均满意度"]]
display(segment_analysis)

,TenureGroup,用户数,占比,流失率,平均订单数,平均返现,平均满意度
0,新用户,508,0.09,0.54,1.89,142.44,3.18
1,0-6个月,1642,0.29,0.26,2.68,164.87,3.09
2,7-12个月,1584,0.28,0.10,2.75,163.31,2.99
3,13-24个月,1467,0.26,0.06,3.70,204.92,3.09
4,24个月以上,429,0.08,0.00,3.55,222.34,3.05


In [7]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：**  
新用户（TenureGroup=新用户）的流失率为53.54%，远高于24个月以上老用户的0%。同时，新用户的平均订单数（1.89单）也明显低于13-24个月用户（3.70单）和24个月以上用户（3.55单）。

**可能解释：**  
新用户可能尚未形成稳定的使用习惯和平台依赖，对服务体验的容忍度较低，因此更容易流失。而使用时长较长的用户可能已与平台建立了信任关系和消费路径依赖，流失风险较低。但这也可能与不同生命周期用户的获取渠道、优惠活动等差异有关，仍需进一步验证。

## 任务4：双维度交叉分析（必做）

In [8]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
).reset_index()

# TODO：新增“样本提示”列；用户数<30标记为“小样本”，否则为“可观察”
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")
# TODO：按流失率或用户数排序并展示
tenure_order = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
cross_analysis["_sort"] = cross_analysis[dim_1].apply(lambda x: tenure_order.index(x))
cross_analysis = cross_analysis.sort_values(["_sort", "流失率"], ascending=[True, False]).drop(columns="_sort").reset_index(drop=True)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
0,新用户,1,194,139,0.72,2.12,可观察
1,新用户,0,314,133,0.42,1.75,可观察
2,0-6个月,1,465,236,0.51,2.87,可观察
3,0-6个月,0,1177,189,0.16,2.61,可观察
4,7-12个月,1,406,81,0.20,2.67,可观察
5,7-12个月,0,1178,75,0.06,2.78,可观察
6,13-24个月,1,414,52,0.13,3.35,可观察
7,13-24个月,0,1053,43,0.04,3.85,可观察
8,24个月以上,0,304,0,0.00,3.75,可观察
9,24个月以上,1,125,0,0.00,3.06,可观察


In [9]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：**  
新用户 + 已投诉（Complain=1）。该组合在所有生命周期×投诉分组中流失率最高。

**该组合的样本量与流失率：**  
用户数194人（标记为"可观察"），流失率71.65%，即每10名投诉过的新用户中约有7人最终流失。

**为什么不能直接下因果结论：**  
1. 本数据为观察性数据，未进行随机分组或实验控制，无法排除混杂变量（如新用户本身忠诚度就低）；
2. 投诉和流失可能由共同原因导致（如订单体验差同时导致投诉和不再使用），而非投诉"导致"流失；
3. 虽然样本量充足（194人），但缺乏投诉类型、处理结果等过程数据，无法建立完整因果链条。因此只能说投诉与流失存在关联，不能断言投诉导致流失。

## 任务5：报表输出与回读验证（必做）

In [10]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path, encoding="utf-8-sig")
    print(f"{filename}: shape={reloaded.shape}")

overall_metrics.csv: shape=(10, 2)
segment_analysis.csv: shape=(5, 7)
cross_analysis.csv: shape=(10, 7)


In [11]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在新用户（TenureGroup=新用户）用户中，流失率指标为53.54%，与24个月以上老用户（流失率0%）相比高出53.54个百分点。对应证据表：segment_analysis.csv。

### 结论2

在0-6个月且已投诉的用户中，流失率为50.75%，而同生命周期未投诉用户的流失率仅为16.06%，两者相差约34.7个百分点。这说明投诉行为与流失风险在当前样本中存在明显关联，尤其在用户生命周期早期阶段更为突出。对应证据表：cross_analysis.csv。

### 结论3

在13-24个月的老用户群体中，流失率仅为6.48%，且平均订单数（3.70单）显著高于新用户（1.89单）。这说明随着用户使用时长增加，留存稳定性逐步提升，老用户已形成了较强的平台依赖和消费习惯。对应证据表：segment_analysis.csv。

### 分析限制

请至少写一条。例如：缺少订单金额和日期，因此不能计算GMV、客单价或时间趋势。
CashbackAmount为返现金额，不等于实际消费金额或销售额，不能将其当作客单价或营收指标使用。
本数据为用户级汇总数据，一行代表一名用户，缺少订单金额和订单日期字段，因此无法计算GMV、客单价，也无法分析月度订单趋势和季节性变化。

### 运营建议与验证方式

请提出一项建议，并说明还需要什么数据或实验来验证效果。
针对0-6个月的新用户建立"首月护航"机制——在注册后30天内安排专属客服主动跟进，提供首单引导和投诉快速处理通道，优先降低新用户群体的投诉率和流失率。
将新用户随机分为实验组（接受"首月护航"服务）和对照组（常规服务），对比两组在注册后30天内的流失率和投诉处理满意度差异。需要补充的数据包括：用户注册渠道、投诉类型与处理时长、实验分组标签；需要追踪的指标包括30天留存率、复购率、NPS评分。

## 拓展任务（选做）

In [12]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）
# 方向1：使用qcut构建订单活跃度分层
# ----------------------------------------------------
# 注意：因OrderCount存在大量重复值（订单数=1的用户占71.7%），
# qcut自动合并为3个有效分层
df["OrderActivityLevel"] = pd.qcut(df["OrderCount"], q=4, 
                                    labels=["低活跃", "中活跃", "高活跃"],
                                    duplicates='drop')

activity_analysis = df.groupby("OrderActivityLevel", observed=True).agg(
    用户数=("CustomerID", "nunique"),
    订单数范围=("OrderCount", lambda x: f"{x.min():.0f}-{x.max():.0f}"),
    平均订单数=("OrderCount", "mean"),
    流失率=("Churn", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
).reset_index()
display(activity_analysis)


,OrderActivityLevel,用户数,订单数范围,平均订单数,流失率,平均满意度
0,低活跃,4034,1-2,1.57,0.17,3.08
1,中活跃,371,3-3,3.00,0.18,2.88
2,高活跃,1225,4-16,7.55,0.14,3.08


## 提交前检查

- [ ] 已填写组号、成员和专题；
- [ ] 已重启内核并从头运行成功；
- [ ] 所有比例表都包含样本量；
- [ ] 三个CSV已导出并回读；
- [ ] 至少3条结论可对应到具体表格；
- [ ] 已写明分析限制和验证建议；
- [ ] 没有把返现写成消费额，没有把相关写成因果。